# ❤️ Heart Disease Prediction — End-to-End ML Project

**Dataset:** UCI Heart Disease Dataset (Kaggle: `redwankarimsony/heart-disease-data`)  
**Task:** Binary Classification — Predict presence of heart disease  
**Author:** Final-Year Engineering Project  

---

## Project Outline
1. Dataset Exploration & EDA
2. Data Preprocessing
3. Feature Engineering
4. Model Training (10 Algorithms)
5. Model Evaluation & Comparison
6. Explainable AI (SHAP)
7. Patient Prediction Module
8. Hyperparameter Tuning
9. Model Saving & Loading
10. Final Report

## 0. Setup & Imports

In [ ]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.inspection import permutation_importance
import shap
import joblib
import os

# XGBoost
from xgboost import XGBClassifier

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✅ All imports successful!')
print(f'NumPy: {np.__version__} | Pandas: {pd.__version__}')

---
## 1. Dataset Loading & Exploration

In [ ]:
# Load dataset (downloads automatically if not present)
import requests, io

DATA_URL = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/heart.csv'

try:
    df = pd.read_csv('data/heart.csv')
    print('✅ Loaded from local file: data/heart.csv')
except FileNotFoundError:
    try:
        response = requests.get(DATA_URL, timeout=10)
        df = pd.read_csv(io.StringIO(response.text))
        os.makedirs('data', exist_ok=True)
        df.to_csv('data/heart.csv', index=False)
        print(f'✅ Downloaded and saved to data/heart.csv')
    except Exception as e:
        print(f'⚠️  Download failed ({e}), generating synthetic data...')
        # Synthetic fallback included via data_loader
        import sys; sys.path.insert(0, '.')
        from utils.data_loader import _generate_synthetic_data
        df = _generate_synthetic_data()

# Normalize target: 0=No Disease, 1=Disease
df['target'] = (df['target'] > 0).astype(int)
print(f'Dataset shape: {df.shape}')
df.head()

### 1.1 Dataset Statistics

In [ ]:
print('=' * 55)
print('DATASET OVERVIEW')
print('=' * 55)
print(f'Records          : {len(df):,}')
print(f'Features         : {df.shape[1] - 1}')
print(f'Target classes   : {df["target"].nunique()} (0=No Disease, 1=Disease)')
print(f'Disease cases    : {df["target"].sum():,} ({df["target"].mean()*100:.1f}%)')
print(f'Healthy cases    : {(df["target"]==0).sum():,} ({(df["target"]==0).mean()*100:.1f}%)')
print(f'Missing values   : {df.isnull().sum().sum()}')
print(f'Duplicate rows   : {df.duplicated().sum()}')
print(f'Age range        : {df["age"].min()} – {df["age"].max()} years')
print(f'Male patients    : {(df["sex"]==1).mean()*100:.1f}%')

In [ ]:
# Statistical summary
df.describe().T.round(3)

In [ ]:
# Data types and null counts
pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.notnull().sum(),
    'null': df.isnull().sum(),
    'unique': df.nunique()
})

---
## 2. Exploratory Data Analysis

In [ ]:
# Target distribution
fig = px.pie(
    df['target'].value_counts().reset_index(),
    values='count', names=df['target'].value_counts().index.map({0: 'No Disease', 1: 'Heart Disease'}),
    title='Target Distribution', hole=0.4,
    color_discrete_sequence=['#2ECC71', '#E84855']
)
fig.show()

In [ ]:
# Age distribution by target
df_plot = df.copy()
df_plot['Status'] = df_plot['target'].map({0: 'No Disease', 1: 'Heart Disease'})

fig = px.histogram(
    df_plot, x='age', color='Status', nbins=20,
    barmode='overlay', opacity=0.75,
    color_discrete_map={'No Disease': '#2ECC71', 'Heart Disease': '#E84855'},
    title='Age Distribution by Heart Disease Status',
    labels={'age': 'Age (years)'}
)
fig.show()

In [ ]:
# Correlation heatmap
corr = df.corr(numeric_only=True)
fig = px.imshow(
    corr, color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='Feature Correlation Heatmap',
    text_auto='.2f', aspect='auto'
)
fig.update_layout(height=550)
fig.show()

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing Values per Feature:')
print(missing[missing > 0] if missing.sum() > 0 else '  ✅ No missing values found!')

# Visualize
fig = px.bar(
    x=df.columns, y=df.isnull().sum().values,
    title='Missing Values per Feature',
    labels={'x': 'Feature', 'y': 'Missing Count'},
    color=df.isnull().sum().values,
    color_continuous_scale='Reds'
)
fig.show()

In [ ]:
# Feature distributions
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()
for i, col in enumerate(df.columns):
    for target_val, color, label in [(0, '#2ECC71', 'No Disease'), (1, '#E84855', 'Heart Disease')]:
        axes[i].hist(df[df['target'] == target_val][col], alpha=0.65,
                     color=color, label=label, bins=15, density=True)
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=8)

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Feature Distributions by Heart Disease Status', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: feature_distributions.png')

In [ ]:
# Pairplot for key features
key_cols = ['age', 'thalach', 'chol', 'oldpeak', 'trestbps', 'target']
df_pair = df[key_cols].copy()
df_pair['Status'] = df_pair['target'].map({0: 'No Disease', 1: 'Heart Disease'})

fig = px.scatter_matrix(
    df_pair, dimensions=['age', 'thalach', 'chol', 'oldpeak', 'trestbps'],
    color='Status',
    color_discrete_map={'No Disease': '#2ECC71', 'Heart Disease': '#E84855'},
    title='Pairplot: Key Continuous Features',
    opacity=0.55
)
fig.update_traces(diagonal_visible=False, marker_size=3)
fig.update_layout(height=700)
fig.show()

---
## 3. Data Preprocessing

In [ ]:
df_processed = df.copy()

# 3.1 Handle Missing Values
print('Before imputation:')
print(df_processed.isnull().sum())

for col in df_processed.columns:
    if df_processed[col].isnull().any():
        if df_processed[col].dtype in [np.float64, np.int64]:
            df_processed[col].fillna(df_processed[col].median(), inplace=True)
        else:
            df_processed[col].fillna(df_processed[col].mode()[0], inplace=True)

print('\nAfter imputation (missing values):')
print(df_processed.isnull().sum())

In [ ]:
# 3.2 Outlier Detection & Capping (IQR)
continuous_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
outlier_report = {}

fig, axes = plt.subplots(2, len(continuous_cols), figsize=(18, 8))

for i, col in enumerate(continuous_cols):
    Q1 = df_processed[col].quantile(0.25)
    Q3 = df_processed[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df_processed[col] < lower) | (df_processed[col] > upper)).sum()
    outlier_report[col] = n_out

    # Before capping
    axes[0][i].boxplot(df_processed[col])
    axes[0][i].set_title(f'{col}\nBefore (outliers={n_out})', fontsize=9)

    df_processed[col] = df_processed[col].clip(lower, upper)

    # After capping
    axes[1][i].boxplot(df_processed[col])
    axes[1][i].set_title(f'{col}\nAfter Capping', fontsize=9)

axes[0][0].set_ylabel('Before Capping', fontweight='bold')
axes[1][0].set_ylabel('After Capping', fontweight='bold')
plt.suptitle('Outlier Detection and Capping (IQR Method)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outlier_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Outliers detected per feature:', outlier_report)

In [ ]:
# 3.3 Feature-Target Split
feature_names = [c for c in df_processed.columns if c != 'target']
X = df_processed[feature_names]
y = df_processed['target']

print(f'Features: {feature_names}')
print(f'X shape: {X.shape} | y shape: {y.shape}')
print(f'Class distribution:\n{y.value_counts()}')

In [ ]:
# 3.4 Stratified Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Train disease rate: {y_train.mean()*100:.1f}% | Test disease rate: {y_test.mean()*100:.1f}%')

# 3.5 Standard Scaling
scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names, index=X_train.index)
X_test_sc = pd.DataFrame(scaler.transform(X_test), columns=feature_names, index=X_test.index)

print('\n✅ Preprocessing complete!')
print('Before scaling (X_train sample):')
display(X_train.describe().T[['mean', 'std']].head())
print('\nAfter scaling (X_train_sc sample):')
display(X_train_sc.describe().T[['mean', 'std']].head())

---
## 4. Model Training

In [ ]:
# Define all 10 models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1),
    'Support Vector Machine': SVC(probability=True, kernel='rbf', C=1.0, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes': GaussianNB(),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                              random_state=42, eval_metric='logloss', verbosity=0),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=4,
                                                      learning_rate=0.1, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, learning_rate=0.5,
                                    random_state=42, algorithm='SAMME'),
    'Neural Network (MLP)': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500,
                                           random_state=42, alpha=0.001, early_stopping=True),
}

print(f'✅ Defined {len(models)} models for training.')

In [ ]:
import time

results = []
trained = {}
curves_data = {}

os.makedirs('models', exist_ok=True)

for name, model in models.items():
    print(f'Training: {name}...', end=' ', flush=True)
    t0 = time.time()

    model.fit(X_train_sc, y_train)
    elapsed = round(time.time() - t0, 3)

    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    roc  = roc_auc_score(y_test, y_prob)
    cv   = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='accuracy')
    cm   = confusion_matrix(y_test, y_pred)

    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1 Score': round(f1, 4),
        'ROC-AUC': round(roc, 4),
        'CV Mean': round(cv.mean(), 4),
        'CV Std': round(cv.std(), 4),
        'Train Time (s)': elapsed,
    })

    trained[name] = {
        'model': model, 'y_pred': y_pred, 'y_prob': y_prob, 'cm': cm,
        'cr': classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']),
    }

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    curves_data[name] = {'fpr': fpr, 'tpr': tpr, 'roc_auc': roc}

    joblib.dump(model, f'models/{name.replace(" ", "_")}.pkl')
    print(f'✅ Acc={acc:.4f} | F1={f1:.4f} | Time={elapsed}s')

results_df = pd.DataFrame(results).set_index('Model')
print('\n🏁 All models trained!')

---
## 5. Evaluation Metrics

In [ ]:
# Complete metrics comparison table
print('=' * 90)
print('ALGORITHM COMPARISON TABLE')
print('=' * 90)
display(results_df.style
    .highlight_max(subset=['Accuracy','Precision','Recall','F1 Score','ROC-AUC','CV Mean'], color='#2a4a2a')
    .highlight_min(subset=['Accuracy','Precision','Recall','F1 Score','ROC-AUC','CV Mean'], color='#4a2a2a')
    .format('{:.4f}', subset=['Accuracy','Precision','Recall','F1 Score','ROC-AUC','CV Mean','CV Std'])
)

In [ ]:
# Best model
best_name = results_df['F1 Score'].idxmax()
worst_name = results_df['F1 Score'].idxmin()
print(f'🥇 Best Model  : {best_name} (F1={results_df.loc[best_name, "F1 Score"]:.4f})')
print(f'🔴 Worst Model : {worst_name} (F1={results_df.loc[worst_name, "F1 Score"]:.4f})')

In [ ]:
# Confusion matrices for top 4 models
top4 = results_df['F1 Score'].nlargest(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, name in zip(axes.flatten(), top4):
    cm = trained[name]['cm']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Disease', 'Disease'],
                yticklabels=['No Disease', 'Disease'],
                annot_kws={'size': 16})
    ax.set_title(f'{name}\nAcc={results_df.loc[name, "Accuracy"]:.4f}  F1={results_df.loc[name, "F1 Score"]:.4f}',
                 fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Top 4 Models', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Classification report for best model
print(f'Classification Report — {best_name}')
print('=' * 60)
print(trained[best_name]['cr'])

---
## 6. Algorithm Comparison Visualizations

In [ ]:
# Bar charts for each metric
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
colors = ['#3A86FF', '#E84855', '#2ECC71', '#F39C12', '#9B59B6']

fig, axes = plt.subplots(1, len(metrics), figsize=(22, 6))
for ax, metric, color in zip(axes, metrics, colors):
    sorted_df = results_df[metric].sort_values(ascending=False)
    bars = ax.barh(sorted_df.index, sorted_df.values, color=color, alpha=0.85)
    ax.set_xlim(0, 1.08)
    ax.set_title(metric, fontweight='bold', fontsize=12)
    for bar, val in zip(bars, sorted_df.values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('Algorithm Comparison — All Metrics', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('algorithm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves for all models
fig = go.Figure()
colors_plotly = px.colors.qualitative.Bold
for i, (name, data) in enumerate(curves_data.items()):
    fig.add_trace(go.Scatter(
        x=data['fpr'], y=data['tpr'], mode='lines',
        name=f'{name} (AUC={data["roc_auc"]:.3f})',
        line=dict(color=colors_plotly[i % len(colors_plotly)], width=2)
    ))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
    line=dict(dash='dash', color='gray'), name='Random'))
fig.update_layout(
    title='ROC Curves — All Models',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    height=550
)
fig.show()

In [ ]:
# Radar chart
radar_metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
fig = go.Figure()
for i, name in enumerate(results_df.index):
    vals = [results_df.loc[name, m] for m in radar_metrics] + [results_df.loc[name, radar_metrics[0]]]
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=radar_metrics + [radar_metrics[0]],
        fill='toself', name=name,
        line=dict(color=colors_plotly[i % len(colors_plotly)]), opacity=0.65
    ))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Radar Chart: Algorithm Comparison',
    height=600
)
fig.show()

In [ ]:
# Algorithm ranking table
ranking = results_df.copy()
ranking['Score'] = (ranking[['Accuracy','Precision','Recall','F1 Score','ROC-AUC']].mean(axis=1))
ranking = ranking.sort_values('Score', ascending=False)
ranking.insert(0, 'Rank', range(1, len(ranking)+1))
print('Algorithm Ranking (Best to Worst):')
display(ranking)

---
## 7. Hyperparameter Tuning (Best Model)

In [ ]:
print(f'Tuning hyperparameters for: {best_name}')

if 'Random Forest' in best_name:
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [4, 6, 8],
        'min_samples_split': [2, 5, 10]
    }
    base_model = RandomForestClassifier(random_state=42, n_jobs=-1)
elif 'XGBoost' in best_name:
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 4, 6],
        'learning_rate': [0.05, 0.1, 0.2]
    }
    base_model = XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0)
elif 'Gradient' in best_name:
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 4, 6],
        'learning_rate': [0.05, 0.1, 0.2]
    }
    base_model = GradientBoostingClassifier(random_state=42)
else:
    param_grid = {'C': [0.1, 1, 10], 'max_iter': [500, 1000]}
    base_model = LogisticRegression(random_state=42)

grid_search = GridSearchCV(
    base_model, param_grid, cv=5,
    scoring='f1', n_jobs=-1, verbose=0
)
grid_search.fit(X_train_sc, y_train)

best_params = grid_search.best_params_
print(f'Best Parameters: {best_params}')
print(f'Best CV F1 Score: {grid_search.best_score_:.4f}')

# Evaluate tuned model
tuned_model = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test_sc)
print(f'Tuned Test Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}')
print(f'Tuned Test F1 Score: {f1_score(y_test, y_pred_tuned):.4f}')

---
## 8. Learning Curves

In [ ]:
best_model_obj = trained[best_name]['model']

train_sizes, train_scores, val_scores = learning_curve(
    best_model_obj, X_train_sc, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='accuracy', n_jobs=-1
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(train_sizes,
    train_scores.mean(1) - train_scores.std(1),
    train_scores.mean(1) + train_scores.std(1),
    alpha=0.15, color='#E84855')
ax.fill_between(train_sizes,
    val_scores.mean(1) - val_scores.std(1),
    val_scores.mean(1) + val_scores.std(1),
    alpha=0.15, color='#3A86FF')
ax.plot(train_sizes, train_scores.mean(1), 'o-', color='#E84855', label='Training Score')
ax.plot(train_sizes, val_scores.mean(1), 'o-', color='#3A86FF', label='Validation Score')
ax.set_xlabel('Training Samples')
ax.set_ylabel('Accuracy')
ax.set_title(f'Learning Curve — {best_name}', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Explainable AI (SHAP)

In [ ]:
print(f'Computing SHAP values for: {best_name}')
best_model_obj = trained[best_name]['model']
X_test_arr = X_test_sc.values

if hasattr(best_model_obj, 'feature_importances_'):
    explainer = shap.TreeExplainer(best_model_obj)
    shap_values = explainer.shap_values(X_test_arr[:100])
    shap_vals = shap_values[1] if isinstance(shap_values, list) else shap_values
    expected_val = explainer.expected_value
    if isinstance(expected_val, list): expected_val = expected_val[1]
else:
    explainer = shap.KernelExplainer(
        best_model_obj.predict_proba, shap.sample(X_train_sc.values, 50)
    )
    shap_values = explainer.shap_values(X_test_arr[:30], nsamples=100)
    shap_vals = shap_values[1] if isinstance(shap_values, list) else shap_values
    expected_val = explainer.expected_value
    if isinstance(expected_val, list): expected_val = expected_val[1]

print('✅ SHAP values computed!')

In [ ]:
# SHAP Summary Plot (Beeswarm)
plt.figure(figsize=(12, 7))
shap.summary_plot(shap_vals, X_test_arr[:len(shap_vals)],
                  feature_names=feature_names, show=False)
plt.title(f'SHAP Summary Plot — {best_name}', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Bar Plot (Global Importance)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_test_arr[:len(shap_vals)],
                  feature_names=feature_names,
                  plot_type='bar', show=False)
plt.title('Global Feature Importance (Mean |SHAP|)', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Waterfall for single patient
PATIENT_IDX = 0
shap_exp = shap.Explanation(
    values=shap_vals[PATIENT_IDX],
    base_values=expected_val,
    data=X_test_arr[PATIENT_IDX],
    feature_names=feature_names
)
plt.figure(figsize=(12, 6))
shap.waterfall_plot(shap_exp, show=False)
actual = y_test.iloc[PATIENT_IDX]
predicted = trained[best_name]['y_pred'][PATIENT_IDX]
plt.title(f'SHAP Waterfall — Patient {PATIENT_IDX} | Actual={actual} | Predicted={predicted}',
          fontweight='bold')
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance from model
if hasattr(best_model_obj, 'feature_importances_'):
    fi = pd.Series(best_model_obj.feature_importances_, index=feature_names)
    fi = fi.sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    fi.plot(kind='barh', ax=ax, color='#E84855', alpha=0.85)
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Feature Importance — {best_name}', fontweight='bold')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

# Permutation Importance
print('Computing permutation importance...')
pi = permutation_importance(best_model_obj, X_test_sc, y_test, n_repeats=15, random_state=42)
pi_df = pd.DataFrame({'Feature': feature_names, 'Importance': pi.importances_mean,
                        'Std': pi.importances_std}).sort_values('Importance', ascending=False)
print('\nPermutation Importance:')
display(pi_df)

---
## 10. Patient Prediction Module

In [ ]:
def predict_heart_disease(patient: dict, model, scaler, feature_names: list):
    """
    Predict heart disease risk for a single patient.
    
    Parameters
    ----------
    patient : dict  — patient clinical values
    model   : trained sklearn model
    scaler  : fitted StandardScaler
    feature_names : list of feature column names
    """
    patient_df = pd.DataFrame([patient])[feature_names]
    patient_sc = scaler.transform(patient_df)
    pred = model.predict(patient_sc)[0]
    prob = model.predict_proba(patient_sc)[0][1]
    risk = 'HIGH' if prob >= 0.65 else 'MODERATE' if prob >= 0.35 else 'LOW'

    print('=' * 55)
    print('HEART DISEASE PREDICTION RESULT')
    print('=' * 55)
    print(f'Prediction    : {"Heart Disease DETECTED" if pred==1 else "No Heart Disease"}')
    print(f'Risk Probability : {prob*100:.1f}%')
    print(f'Risk Level    : {risk}')
    print('=' * 55)

    if hasattr(model, 'feature_importances_'):
        top_feats = sorted(zip(feature_names, model.feature_importances_),
                           key=lambda x: x[1], reverse=True)[:5]
        print('Top Contributing Features:')
        for i, (feat, imp) in enumerate(top_feats, 1):
            print(f'  {i}. {feat}: {imp:.4f}')

    return int(pred), float(prob), risk

# Example: High-Risk Patient
high_risk_patient = {
    'age': 67, 'sex': 1, 'cp': 0, 'trestbps': 160,
    'chol': 286, 'fbs': 0, 'restecg': 0, 'thalach': 108,
    'exang': 1, 'oldpeak': 1.5, 'slope': 1, 'ca': 3, 'thal': 2
}

pred, prob, risk = predict_heart_disease(
    high_risk_patient, trained[best_name]['model'], scaler, feature_names
)

In [ ]:
# Low-risk patient example
low_risk_patient = {
    'age': 45, 'sex': 0, 'cp': 2, 'trestbps': 118,
    'chol': 195, 'fbs': 0, 'restecg': 1, 'thalach': 172,
    'exang': 0, 'oldpeak': 0.0, 'slope': 2, 'ca': 0, 'thal': 0
}

predict_heart_disease(
    low_risk_patient, trained[best_name]['model'], scaler, feature_names
)

---
## 11. Model Saving & Loading

In [ ]:
# Save best model and scaler
joblib.dump(trained[best_name]['model'], f'models/best_model_{best_name.replace(" ", "_")}.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
print(f'✅ Saved: models/best_model_{best_name.replace(" ", "_")}.pkl')
print('✅ Saved: models/scaler.pkl')

# Load and verify
loaded_model = joblib.load(f'models/best_model_{best_name.replace(" ", "_")}.pkl')
loaded_scaler = joblib.load('models/scaler.pkl')
y_pred_loaded = loaded_model.predict(loaded_scaler.transform(X_test_sc))
print(f'\n✅ Loaded model accuracy: {accuracy_score(y_test, y_pred_loaded):.4f} (matches original)')

---
## 12. Final Report

In [ ]:
best_row = results_df.loc[best_name]

print('=' * 65)
print('FINAL REPORT — Heart Disease Prediction')
print('=' * 65)
print(f'Dataset          : UCI Heart Disease ({len(df)} records, {len(feature_names)} features)')
print(f'Best Algorithm   : {best_name}')
print(f'Accuracy         : {best_row["Accuracy"]*100:.2f}%')
print(f'Precision        : {best_row["Precision"]*100:.2f}%')
print(f'Recall (Sensitivity): {best_row["Recall"]*100:.2f}%')
print(f'F1 Score         : {best_row["F1 Score"]:.4f}')
print(f'ROC-AUC          : {best_row["ROC-AUC"]:.4f}')
print(f'5-Fold CV        : {best_row["CV Mean"]:.4f} ± {best_row["CV Std"]:.4f}')
print('=' * 65)
print('Algorithm Ranking:')
for rank, (name, row) in enumerate(ranking.iterrows(), 1):
    if isinstance(row, pd.Series):
        f1_val = row.get('F1 Score', results_df.loc[name, 'F1 Score'])
    else:
        f1_val = results_df.loc[name, 'F1 Score']
    marker = '🥇' if rank == 1 else '🥈' if rank == 2 else '🥉' if rank == 3 else '  '
    print(f'{marker} {rank:2d}. {name:<30} F1={results_df.loc[name, "F1 Score"]:.4f}')
print('=' * 65)
print('Key Features (by importance):')
if hasattr(trained[best_name]['model'], 'feature_importances_'):
    for i, (feat, imp) in enumerate(sorted(
        zip(feature_names, trained[best_name]['model'].feature_importances_),
        key=lambda x: x[1], reverse=True
    )[:6], 1):
        print(f'  {i}. {feat:15s}: {imp:.4f}')
print('=' * 65)
print('Future Scope:')
print('  1. Deep Learning on ECG time-series data')
print('  2. Federated Learning across hospital networks')
print('  3. Multi-class severity prediction (0-4 scale)')
print('  4. Real-time EHR integration via HL7/FHIR')
print('  5. Mobile app for point-of-care assessment')
print('=' * 65)